## Data Prep for Multi-Modal RAG 

### Installing Utilities and Libraries

In [ ]:
%pip install numpy==1.26.4 openai==1.69.0 langchain-community==0.4.1 PyPDF2==3.0.1 anthropic==0.120.2

### Setting up the Environment

In [ ]:
import os
from dotenv import load_dotenv
import os
import base64
import mimetypes
import pandas as pd

from anthropic import Anthropic
from PyPDF2 import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

claude_model_name = os.getenv("CLAUDE_MODEL_NAME")
claude_api_key = os.getenv("CLAUDE_API_KEY")

### Create the Anthropic Client

In [ ]:
import anthropic

client = anthropic.Anthropic(api_key=claude_api_key)

### Verify and List the Dataset

In [ ]:
knowledge_folder = "./knowledge"
docs_folder = "./knowledge/docs"
images_folder = "./knowledge/images"

print("Knowledge folder:")
print(os.listdir(knowledge_folder))

print("\nImages:")
print(os.listdir(images_folder))

print("\nDocuments:")
print(os.listdir(docs_folder))

### Define the Base64 Conversion Function for Images

In [ ]:
def encode_image(image_path):
    """Convert an image into a Base64 encoded string."""

    with open(image_path, "rb") as image_file:
        return base64.b64encode(
            image_file.read()
        ).decode("utf-8")

### Process all Images and Create an Array

In [ ]:
image_records = []

for filename in os.listdir(images_folder):

    if filename.lower().endswith(".png"):

        image_path = os.path.join(
            images_folder,
            filename
        )

        image_records.append(
            {
                "content_path": image_path,
                "base64_content": encode_image(image_path),
            }
        )


images_metadata = pd.DataFrame(image_records)

print(f"Images processed: {len(images_metadata)}")

images_metadata.head()

### Create the Image Verbalization Function

In [ ]:
def verbalize_image(image_path):

    image_data = encode_image(image_path)

    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=500,

        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "source": {
                            "type": "base64",
                            "media_type": "image/png",
                            "data": image_data,
                        },
                    },
                    {
                        "type": "text",
                        "text": (
                            "Describe this image in detail for use "
                            "in a retrieval-augmented generation "
                            "knowledge base. Capture important "
                            "concepts, objects, labels, relationships, "
                            "and textual information."
                        ),
                    },
                ],
            }
        ],
    )

    return response.content[0].text

### Process all Images with Verbalization

In [ ]:
image_verbalizations = []

for _, row in images_metadata.iterrows():

    print(f"Processing: {row['content_path']}")

    description = verbalize_image(
        row["content_path"]
    )

    image_verbalizations.append(
        {
            "content_path": row["content_path"],
            "chunk": description,
            "content_type": "image",
        }
    )


images_verbalization = pd.DataFrame(
    image_verbalizations
)

images_verbalization.head()

### Create Recursive Chunking Function for PDFs

In [ ]:
def perform_fixed_size_chunking(
    document,
    chunk_size=2000,
    chunk_overlap=500,
):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,

        separators=[
            "\n\n",
            "\n",
            ". ",
            " ",
            "",
        ],
    )

    return text_splitter.split_text(
        document
    )

### Process PDFs with Chunking

In [ ]:
document_chunks = []


for filename in os.listdir(docs_folder):

    file_path = os.path.join(
        docs_folder,
        filename
    )

    if (
        os.path.isfile(file_path)
        and filename.lower().endswith(".pdf")
    ):

        print(
            f"Processing: {filename}"
        )

        reader = PdfReader(
            file_path
        )

        text = ""

        for page in reader.pages:

            page_text = page.extract_text()

            if page_text:
                text += page_text + "\n"

        # Only process documents
        # containing extracted text
        if text.strip():

            chunks = perform_fixed_size_chunking(
                text
            )

            for i, chunk in enumerate(chunks):

                if chunk.strip():

                    document_chunks.append(
                        {
                            "content_path":
                                file_path,

                            "chunk":
                                chunk,

                            "content_type":
                                "document",
                        }
                    )

In [ ]:
docs_chunks = pd.DataFrame(
    document_chunks
)

print(
    f"Total document chunks: "
    f"{len(docs_chunks)}"
)

docs_chunks.head()

### Check number of Chunks per PDF

In [ ]:
docs_chunks.groupby(
    "content_path"
).size().reset_index(
    name="chunk_count"
)

### Combine Image and Document Chunks

In [ ]:
final_rag_dataset = pd.concat(
    [
        images_verbalization,
        docs_chunks,
    ],
    ignore_index=True,
)

### Create final RAG Dataset with unique IDs

In [ ]:
final_rag_dataset.insert(
    0,
    "id",
    range(1, len(final_rag_dataset) + 1)
)

In [ ]:
final_rag_dataset.head()

### Save it to a JSON File

In [ ]:
final_rag_dataset.to_json(
    "final_rag_dataset.json",
    orient="records",
    indent=4,
    force_ascii=False
)

print("Dataset saved to final_rag_dataset.json")